In [596]:
import torch
from torch import nn, optim
from torch.nn.functional import mse_loss
from tqdm import tqdm, trange

device = torch.device("cpu" if torch.cuda.is_available() else "cpu")

state_dim = 32
message_dim = 32
action_dim = 3
hidden_dim = 64

In [ ]:
class MessageActor(nn.Module):
    def __init__(self, state_dim: int, message_dim: int, hidden_dim: int = 256) -> None:
        super().__init__()
        self._net = nn.Sequential(
            nn.Linear(state_dim + 2 * message_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, message_dim),
        )

    def forward(
        self,
        state: torch.Tensor,
        prev_message: torch.Tensor,
        input_message: torch.Tensor,
    ) -> torch.Tensor:
        return self._net(torch.cat([state, prev_message, input_message], dim = -1))


class FinalActor(nn.Module):
    def __init__(self, state_dim: int, message_dim: int, action_dim: int, hidden_dim: int = 256) -> None:
        super().__init__()
        self._net = nn.Sequential(
            nn.Linear(state_dim + 2 * message_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, action_dim),
        )

    def forward(
        self,
        state: torch.Tensor,
        prev_message: torch.Tensor,
        input_message: torch.Tensor,
    ) -> torch.Tensor:
        state = torch.zeros_like(state)
        return self._net(torch.cat([state, prev_message, input_message], dim = -1))


class MessageCritic(nn.Module):
    def __init__(self, state_dim: int, message_dim: int, hidden_dim: int = 256) -> None:
        super().__init__()
        self._net = nn.Sequential(
            nn.Linear(state_dim + 4 * message_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(
        self,
        state: torch.Tensor,
        prev_prev_message: torch.Tensor,
        prev_message: torch.Tensor,
        input_message: torch.Tensor,
        output_message: torch.Tensor,
    ) -> torch.Tensor:
        return self._net(torch.cat(
            [state, prev_prev_message, prev_message, input_message, output_message],
            dim=-1,
        ))


class FinalCritic(nn.Module):
    def __init__(self, state_dim: int, message_dim: int, action_dim: int, hidden_dim: int = 256) -> None:
        super().__init__()
        self._net = nn.Sequential(
            nn.Linear(state_dim + 2 * message_dim + action_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(
        self,
        state: torch.Tensor,
        prev_message: torch.Tensor,
        input_message: torch.Tensor,
        action: torch.Tensor,
    ) -> torch.Tensor:
        return self._net(torch.cat([state, prev_message, input_message, action], dim=-1))

In [598]:
message_actor1, target_message_actor1 = (MessageActor(state_dim, message_dim, hidden_dim) for _ in range(2))
message_actor2, target_message_actor2 = (MessageActor(state_dim, message_dim, hidden_dim) for _ in range(2))
final_actor, target_final_actor = (FinalActor(state_dim, message_dim, action_dim, hidden_dim) for _ in range(2))
message_critic1_h1, target_message_critic1_h1 = (MessageCritic(state_dim, message_dim, hidden_dim) for _ in range(2))
message_critic1_h2, target_message_critic1_h2 = (MessageCritic(state_dim, message_dim, hidden_dim) for _ in range(2))
message_critic2_h1, target_message_critic2_h1 = (MessageCritic(state_dim, message_dim, hidden_dim) for _ in range(2))
message_critic2_h2, target_message_critic2_h2 = (MessageCritic(state_dim, message_dim, hidden_dim) for _ in range(2))
final_critic_h1, target_final_critic_h1 = (FinalCritic(state_dim, message_dim, action_dim, hidden_dim) for _ in range(2))
final_critic_h2, target_final_critic_h2 = (FinalCritic(state_dim, message_dim, action_dim, hidden_dim) for _ in range(2))

online_models = [
    message_actor1,
    message_actor2,
    final_actor,
    message_critic1_h1,
    message_critic1_h2,
    message_critic2_h1,
    message_critic2_h2,
    final_critic_h1,
    final_critic_h2,
]
target_models = [
    target_message_actor1,
    target_message_actor2,
    target_final_actor,
    target_message_critic1_h1,
    target_message_critic1_h2,
    target_message_critic2_h1,
    target_message_critic2_h2,
    target_final_critic_h1,
    target_final_critic_h2,
]

for online, target in zip(online_models, target_models):
    target.load_state_dict(online.state_dict())
    for param in target.parameters():
        param.requires_grad_(False)

for model in online_models + target_models:
    model.to(device)

In [604]:
def reward_fn(state: torch.Tensor, action: torch.Tensor) -> torch.Tensor:
    with torch.no_grad():
        reward = 100
        reward = reward - mse_loss(state.norm(p=2.5, dim=-1), action[:, 0], reduction="none")
        reward = reward - mse_loss(state.norm(p=2, dim=-1), action[:, 1], reduction="none")
        reward = reward - mse_loss(state.norm(p=1.5, dim=-1), action[:, 2], reduction="none")
        return reward.clamp(min=-100.0).unsqueeze(-1)

In [605]:
def polyak_update(target: nn.Module, policy: nn.Module, tau: float) -> None:
    for target_param, policy_param in zip(target.parameters(), policy.parameters()):
        target_param.data.copy_(tau * policy_param.data + (1 - tau) * target_param.data)

In [606]:
lr = 3e-4
message_actor1_opt = optim.Adam(message_actor1.parameters(), lr = lr)
message_actor2_opt = optim.Adam(message_actor2.parameters(), lr = lr)
final_actor_opt = optim.Adam(final_actor.parameters(), lr = lr)
message_critic1_opt = optim.Adam(
    list(message_critic1_h1.parameters()) + list(message_critic1_h2.parameters()),
    lr = lr,
)
message_critic2_opt = optim.Adam(
    list(message_critic2_h1.parameters()) + list(message_critic2_h2.parameters()),
    lr = lr,
)
final_critic_opt = optim.Adam(
    list(final_critic_h1.parameters()) + list(final_critic_h2.parameters()),
    lr = lr,
)

In [607]:
epsilon = 0.05

In [609]:
num_steps = 1000000
batch_size = 128
epsilon_decay = 0.9995
epsilon_min = 0.0001
tau = 0.0001
lambda_error = 0.2
error_max = 50.0
gamma = 0.8
no_message = torch.zeros(batch_size, message_dim)
for step in trange(num_steps):
    state = torch.randn(batch_size, state_dim, requires_grad=True)

    if step % 4 == 0:
        # Train message actor 1
        message1 = message_actor1(state, no_message, no_message)
        message1 = message1 + torch.randn_like(message1) * epsilon
        message2 = target_message_actor2(state, no_message, message1)
        message2 = message2 + torch.randn_like(message2) * epsilon
        reward2_h1 = target_message_critic2_h1(state, no_message, no_message, message1, message2)
        reward2_h2 = target_message_critic2_h2(state, no_message, no_message, message1, message2)
        reward2_min = torch.minimum(reward2_h1, reward2_h2)
        action = target_final_actor(torch.zeros_like(state), message1, message2)
        reward_final_h1 = target_final_critic_h1(state, message1, message2, action)
        reward_final_h2 = target_final_critic_h2(state, message1, message2, action)
        reward_final_min = torch.minimum(reward_final_h1, reward_final_h2)
        reward_true = reward_fn(state, action)
        reward_final_error = mse_loss(reward_final_h1, reward_true, reduction="none")
        reward_final_error = reward_final_error + mse_loss(reward_final_h2, reward_true, reduction="none")
        reward_final_error = reward_final_error.clamp(max=error_max)
        reward_target = (1 - gamma) * reward_true + gamma * (reward_final_min - lambda_error * reward_final_error)
        reward2_error = mse_loss(reward2_h1, reward_target, reduction="none")
        reward2_error = reward2_error + mse_loss(reward2_h2, reward_target, reduction="none")
        reward2_error = reward2_error.clamp(max=error_max)
        message_actor1_loss = (lambda_error * reward2_error - reward2_min).mean()
        message_actor1_opt.zero_grad()
        message_actor1_loss.backward()
        message_actor1_opt.step()
        polyak_update(target_message_actor1, message_actor1, tau)

        # Train message actor 2
        with torch.no_grad():
            message1 = target_message_actor1(state, no_message, no_message)
            message1 = message1 + torch.randn_like(message1) * epsilon
        message1 = message1.detach().clone().requires_grad_(True)
        message2 = message_actor2(state, no_message, message1)
        message2 = message2 + torch.randn_like(message2) * epsilon
        action = target_final_actor(torch.zeros_like(state), message1, message2)
        reward_final_h1 = target_final_critic_h1(state, message1, message2, action)
        reward_final_h2 = target_final_critic_h2(state, message1, message2, action)
        reward_final_min = torch.minimum(reward_final_h1, reward_final_h2)
        reward_true = reward_fn(state, action)
        reward_final_error = mse_loss(reward_final_h1, reward_true, reduction="none")
        reward_final_error = reward_final_error + mse_loss(reward_final_h2, reward_true, reduction="none")
        reward_final_error = reward_final_error.clamp(max=error_max)
        message_actor2_loss = (lambda_error * reward_final_error - reward_final_min).mean()
        message_actor2_opt.zero_grad()
        message_actor2_loss.backward()
        message_actor2_opt.step()
        polyak_update(target_message_actor2, message_actor2, tau)

        # # Train final actor
        with torch.no_grad():
            message1 = target_message_actor1(state, no_message, no_message)
            message1 = message1 + torch.randn_like(message1) * epsilon
            message2 = target_message_actor2(state, no_message, message1)
            message2 = message2 + torch.randn_like(message2) * epsilon
        message1 = message1.detach().clone().requires_grad_(True)
        message2 = message2.detach().clone().requires_grad_(True)
        action = final_actor(torch.zeros_like(state), message1, message2)
        reward_final_h1 = target_final_critic_h1(state, message1, message2, action)
        reward_final_h2 = target_final_critic_h2(state, message1, message2, action)
        reward_final_min = torch.minimum(reward_final_h1, reward_final_h2)
        reward_true = reward_fn(state, action)
        reward_final_error = mse_loss(reward_final_h1, reward_true, reduction="none")
        reward_final_error = reward_final_error + mse_loss(reward_final_h2, reward_true, reduction="none")
        reward_final_error = reward_final_error.clamp(max=error_max)
        final_actor_loss = (lambda_error * reward_final_error - reward_final_min).mean()
        final_actor_opt.zero_grad()
        final_actor_loss.backward()
        final_actor_opt.step()
        polyak_update(target_final_actor, final_actor, tau)

    # Train message critic 2
    with torch.no_grad():
        message1 = target_message_actor1(state, no_message, no_message)
        message1 = message1 + torch.randn_like(message1) * epsilon
    message1 = message1.detach().clone().requires_grad_(True)
    message2 = message_actor2(state, no_message, message1)
    message2 = message2 + torch.randn_like(message2) * epsilon
    reward2_h1 = message_critic2_h1(state, no_message, no_message, message1, message2)
    reward2_h2 = message_critic2_h2(state, no_message, no_message, message1, message2)
    action = target_final_actor(torch.zeros_like(state), message1, message2)
    reward_final_h1 = target_final_critic_h1(state, message1, message2, action)
    reward_final_h2 = target_final_critic_h2(state, message1, message2, action)
    reward_final_min = torch.minimum(reward_final_h1, reward_final_h2)
    reward_true = reward_fn(state, action)
    reward_final_error = mse_loss(reward_final_h1, reward_true, reduction="none")
    reward_final_error = reward_final_error + mse_loss(reward_final_h2, reward_true, reduction="none")
    reward_final_error = reward_final_error.clamp(max=error_max)
    reward_target = (1 - gamma) * reward_true + gamma * (reward_final_min - lambda_error * reward_final_error)
    message_critic2_loss = mse_loss(reward2_h1, reward_target) + mse_loss(reward2_h2, reward_target)
    message_critic2_opt.zero_grad()
    message_critic2_loss.backward()
    message_critic2_opt.step()
    polyak_update(target_message_critic2_h1, message_critic2_h1, tau)
    polyak_update(target_message_critic2_h2, message_critic2_h2, tau)

    # Train final critic
    with torch.no_grad():
        message1 = target_message_actor1(state, no_message, no_message)
        message1 = message1 + torch.randn_like(message1) * epsilon
        message2 = target_message_actor2(state, no_message, message1)
        message2 = message2 + torch.randn_like(message2) * epsilon
    message1 = message1.detach().clone().requires_grad_(True)
    message2 = message2.detach().clone().requires_grad_(True)
    action = final_actor(torch.zeros_like(state), message1, message2)
    reward_final_h1 = final_critic_h1(state, message1, message2, action)
    reward_final_h2 = final_critic_h2(state, message1, message2, action)
    reward_true = reward_fn(state, action)
    final_critic_opt.zero_grad()
    final_critic_loss = mse_loss(reward_final_h1, reward_true) + mse_loss(reward_final_h2, reward_true)
    final_critic_loss.backward()
    final_critic_opt.step()
    polyak_update(target_final_critic_h1, final_critic_h1, tau)
    polyak_update(target_final_critic_h2, final_critic_h2, tau)

    epsilon = max(epsilon * epsilon_decay, epsilon_min)

    if step % 100 == 0:
        print(f"Step {step}, eps: {epsilon:.8f}, ma1l: {message_actor1_loss:.8f}, ma2l: {message_actor2_loss:.8f}, fal: {final_actor_loss:.8f}, mc2l: {message_critic2_loss:.8f}, fcl: {final_critic_loss:.8f}, rw: {reward_true.mean():.8f}")

  0%|          | 0/1000000 [00:00<?, ?it/s]

  0%|          | 4/1000000 [00:00<7:57:04, 34.94it/s]

Step 0, eps: 0.04558141, ma1l: 8.70008469, ma2l: 8.93010330, fal: 9.67232609, mc2l: 257.45245361, fcl: 6106.07812500, rw: -51.73503876


  0%|          | 108/1000000 [00:03<6:53:02, 40.35it/s]

Step 100, eps: 0.04335783, ma1l: 8.70703697, ma2l: 8.66454792, fal: 9.74889755, mc2l: 218.57290649, fcl: 5895.90966797, rw: -53.07186890


  0%|          | 208/1000000 [00:05<6:25:02, 43.28it/s]

... (161 outputs left out)

  1%|          | 8307/1000000 [03:36<7:16:45, 37.84it/s]

Step 8300, eps: 0.00071782, ma1l: 12.32276726, ma2l: -12.92158794, fal: -11.38068008, mc2l: 7.49436331, fcl: 32.95449066, rw: 81.38220215


  1%|          | 8396/1000000 [03:39<7:11:27, 38.30it/s]


KeyboardInterrupt: 

In [612]:
# List of epsilon values to test
epsilons = [0.001, 0.005, 0.01, 0.025, 0.05, 0.1]

# Initialize state (same batch size as your training loop)
state = torch.randn(batch_size, state_dim, device=device)

with torch.no_grad():
    for eps in epsilons:
        # Generate the first message from the actor
        message1 = target_message_actor1(state, no_message, no_message)
        message1 = message1 + torch.randn_like(message1) * eps  # Add noise to message1
        
        # Generate the second message from the target actor
        message2 = target_message_actor2(state, no_message, message1)
        message2 = message2 + torch.randn_like(message2) * eps  # Add noise to message2
        message2 = message2[0].unsqueeze(0).repeat(batch_size, 1)
        
        # Compute action from final actor using noisy messages
        action = final_actor(torch.zeros_like(state), message1, message2)
        
        # Compute reward for final actor
        reward_true = reward_fn(state, action)
        
        # Print the reward for this epsilon
        print(f'Epsilon: {eps:.4f}, Reward: {reward_true.mean().item():.4f}, Abs Message1: {abs(message1).mean().item():.4f}')

Epsilon: 0.0010, Reward: 85.8247, Abs Message1: 0.5583
Epsilon: 0.0050, Reward: 85.9164, Abs Message1: 0.5584
Epsilon: 0.0100, Reward: 85.9975, Abs Message1: 0.5586
Epsilon: 0.0250, Reward: 86.5126, Abs Message1: 0.5589
Epsilon: 0.0500, Reward: 87.1841, Abs Message1: 0.5587
Epsilon: 0.1000, Reward: 85.1901, Abs Message1: 0.5655


So I have to give a more concise specification. There are clearly things that you do not understand:



Starting off with constraints. Constraints are things that are added to the system that wouldn't work in regular RL, but work here because the training isn't rigid here. Meaning they will adapt given the constraints. Having constraints can really help it adapt well. In this case, there is minimum and maximum lifespan. Across a batch the average of the different routes we 80% choose the best route 15% choose the second best route and 5% choose the worst route, excluding the submission route, unless submission is our best pick and the lifespan is in range we submit or unless we must submit (we reached the maximum lifespan). Ordinarily hardcoding an routing policy wouldn't work at all for RL, but note what I said. The agents are learning a general behavior, and will work with constraints. Of course, if this meta-model existed in an RL setup, we wouldn't be doing this for the outside environment, but the agents still will follow constraints as they are encoding a general behavior that will help it in the outside environment.



Now onto the training logic, we want to train on batches. We can only train one agent at a time and collect samples for that agent. This means that agent will use online models at that time. It is slow technically but we don't have much of a choice. We wait to collect a certain number of samples (say 128) before updating the agent. We can update all 3 models. Each agent trains on all possible routes. This is because we train in a batch so not doing so would be wasting training. Error minimization is important. Once we calculate the next state we must calculate the error as well by propagating one time step. We cannot infinitely propagate the error .